# 02 · Evaluation
Evaluates every finished run on its test split. Runs evaluated with an older metric version are re-evaluated automatically.
- `ss`: single-scale inference (primary, all runs).
- SQ3 resolution analysis, for random-split, native-scale SegFormer-B1 and DeepLabV3+ runs (all losses):
  - `f10`: scale 1.0 + horizontal flip (isolates the flip effect)
  - `s15`: scale 1.5 only
  - `s20`: scale 2.0 only
  - `ms`: scales 1.0, 1.5, 2.0 + flip (pre-declared condition)
  `f10`, `s15` and `s20` were added after the first `ms` results (stem IoU decreased with `ms`), to locate which factor causes the change; they are reported as a post-hoc decomposition.
- Saves prediction PNGs for seed 0 (for qualitative figures).

Metrics are computed at native resolution against the original masks: per-class IoU, stem clDice, stem skeleton recall, stem Betti-0 error, stem Boundary IoU (2-px band; default 2%-diagonal band kept for reference).

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
import glob, json
from gwfss_stem import engine
done = sorted(os.path.dirname(p) for p in glob.glob(f'{RESULTS}/*/*/*/*/*/final.pt'))
print(len(done), 'finished runs')
SQ3_TAGS = {                      # tag: (scales, flip)
    'f10': ((1.0,), True),
    's15': ((1.5,), False),
    's20': ((2.0,), False),
    'ms':  ((1.0, 1.5, 2.0), True),
}
for d in done:
    cfg = json.load(open(f'{d}/config.json'))['config']
    engine.evaluate_run(d, DATA_ROOT, SPLIT_FILE, tag='ss', save_preds=cfg['seed'] == 0)
    if cfg['split'] == 'random' and cfg['scale'] == 1.0 and cfg['backbone'] in ('segformer_b1', 'deeplabv3plus_r101'):
        for tag, (scales, flip) in SQ3_TAGS.items():
            engine.evaluate_run(d, DATA_ROOT, SPLIT_FILE, tag=tag, scales=scales, flip=flip)